In [ ]:
# SupplyChain Sentinel — Data Preprocessing

# This notebook prepares the DataCo Smart Supply Chain dataset
# for machine learning.

# Objective:
# Predict whether a shipment is at risk of late delivery using
# only information that would reasonably be available when the
# order is placed.

# Target:
# `Late_delivery_risk`

# Approach:
# 1. Load and inspect the raw data
# 2. Clean the dataset
# 3. Engineer time-based features
# 4. Prevent data leakage
# 5. Create a chronological train/test split
# 6. Build a reusable preprocessing pipeline

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import joblib

# Project root
PROJECT_ROOT = Path.cwd().parent

# Dataset path
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DataCoSupplyChainDataset.csv"
)

# Load the dataset
df = pd.read_csv(
    DATA_PATH,
    encoding="latin1"
)

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully!
Rows: 180,519
Columns: 53


In [ ]:
# Step 9.1 — Create a working copy

df_clean = df.copy()

print("Original shape:", df.shape)
print("Working copy:", df_clean.shape)

Original shape: (180519, 53)
Working copy: (180519, 53)


In [ ]:
# Convert common text placeholders into actual missing values

missing_tokens = ["", "NULL", "null", "N/A", "n/a"]

df_clean = df_clean.replace(
    missing_tokens,
    np.nan
)
# makes missing-value handling consistent, We won't fill missing values yet

In [ ]:
duplicate_count = df_clean.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df_clean = df_clean.drop_duplicates()

print("Shape after duplicate check:", df_clean.shape)

Exact duplicate rows: 0
Shape after duplicate check: (180519, 53)


In [ ]:
drop_cols = [
    # Completely empty or highly incomplete
    "Product Description",
    "Order Zipcode",

    # Personal information
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Password",
    "Customer Street",
    "Customer Zipcode",
    "Customer Id",
    "Order Customer Id",

    # Record and product identifiers
    "Order Id",
    "Order Item Id",
    "Order Item Cardprod Id",
    "Product Card Id",
    "Category Id",
    "Department Id",
    "Product Category Id",

    # Direct outcome leakage
    "Delivery Status",
    "Days for shipping (real)",
    "shipping date (DateOrders)",
    "Order Status",

    # Post-order or potentially outcome-derived fields
    "Benefit per order",
    "Sales per customer",
    "Order Item Profit Ratio",
    "Order Profit Per Order",

    # Redundant or unsuitable fields
    "Product Price",
    "Sales",
    "Order Item Total",
    "Product Image",
    "Product Name",
    "Product Status",

    # Raw coordinates: not needed for this first model
    "Latitude",
    "Longitude",
]

# Keep only columns that actually exist
drop_cols = [
    col for col in drop_cols
    if col in df_clean.columns
]

df_clean = df_clean.drop(columns=drop_cols)

print("Columns dropped:", len(drop_cols))
print("Remaining shape:", df_clean.shape)
print("Remaining columns:", df_clean.columns.tolist())

Columns dropped: 33
Remaining shape: (180519, 20)
Remaining columns: ['Type', 'Days for shipment (scheduled)', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order City', 'Order Country', 'order date (DateOrders)', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Quantity', 'Order Region', 'Order State', 'Shipping Mode']


In [ ]:
# The raw order date contains useful information, but a timestamp isn't always the most convenient input for our first model

# Parse order date
date_col = "order date (DateOrders)"

df_clean[date_col] = pd.to_datetime(
    df_clean[date_col],
    errors="coerce"
)

# Extract date features
df_clean["Order_Month"] = (
    df_clean[date_col].dt.month
)

df_clean["Order_DayOfWeek"] = (
    df_clean[date_col].dt.dayofweek
)

df_clean["Order_Day"] = (
    df_clean[date_col].dt.day
)

# Keep the original timestamp for chronological splitting
print(df_clean[date_col].min())
print(df_clean[date_col].max())

2015-01-01 00:00:00
2018-01-31 23:38:00


In [ ]:
missing_after_drop = (
    df_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_after_drop[
    missing_after_drop > 0
]

Series([], dtype: int64)

In [ ]:
TARGET = "Late_delivery_risk"

print(df_clean[TARGET].value_counts(dropna=False))
print("Missing targets:", df_clean[TARGET].isna().sum())

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64
Missing targets: 0


In [ ]:
df_clean = df_clean.dropna(
    subset=[TARGET]
).copy()

df_clean[TARGET] = df_clean[TARGET].astype(int)

assert set(df_clean[TARGET].unique()).issubset({0, 1})

print("Final cleaned shape:", df_clean.shape)

Final cleaned shape: (180519, 23)


In [ ]:
feature_cols = [
    # Shipment configuration
    "Type",
    "Shipping Mode",
    "Days for shipment (scheduled)",

    # Order timing
    "Order_Month",
    "Order_DayOfWeek",
    "Order_Day",

    # Product and transaction details
    "Category Name",
    "Department Name",
    "Order Item Product Price",
    "Order Item Quantity",
    "Order Item Discount",
    "Order Item Discount Rate",

    # Customer segment
    "Customer Segment",

    # Geographic and market context
    "Market",
    "Order Region",
    "Order Country",
    "Order State",
]

# Keep only features that exist in the cleaned dataset
feature_cols = [
    col for col in feature_cols
    if col in df_clean.columns
]

X = df_clean[feature_cols].copy()
y = df_clean[TARGET].copy()

print("Feature count:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())

Feature count: 17
X shape: (180519, 17)
y shape: (180519,)


,Type,Shipping Mode,Days for shipment (scheduled),Order_Month,Order_DayOfWeek,Order_Day,Category Name,Department Name,Order Item Product Price,Order Item Quantity,Order Item Discount,Order Item Discount Rate,Customer Segment,Market,Order Region,Order Country,Order State
0,DEBIT,Standard Class,4,1,2,31,Sporting Goods,Fitness,327.75,1,13.110000,0.04,Consumer,Pacific Asia,Southeast Asia,Indonesia,Java Occidental
1,TRANSFER,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,16.389999,0.05,Consumer,Pacific Asia,South Asia,India,Rajastán
2,CASH,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,18.030001,0.06,Consumer,Pacific Asia,South Asia,India,Rajastán
3,DEBIT,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,22.940001,0.07,Home Office,Pacific Asia,Oceania,Australia,Queensland
4,PAYMENT,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,29.500000,0.09,Corporate,Pacific Asia,Oceania,Australia,Queensland


In [ ]:
PROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "supply_chain_cleaned.csv"
)

df_clean.to_csv(
    PROCESSED_PATH,
    index=False
)

print(f"Saved cleaned data to: {PROCESSED_PATH}")

Saved cleaned data to: /Users/Kushagra/Desktop/Code/LogiCast/data/processed/supply_chain_cleaned.csv


In [ ]:
FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "feature_columns.txt"
)

FEATURES_PATH.write_text(
    "\n".join(feature_cols)
)

print("Feature list saved.")

Feature list saved.


In [ ]:
# Directory for saving trained models and preprocessing objects

MODEL_DIR = (
    PROJECT_ROOT
    / "src"
    / "models"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Model directory:", MODEL_DIR)

Model directory: /Users/Kushagra/Desktop/Code/LogiCast/src/models


In [ ]:
print("Cleaned dataset:", df_clean.shape)
print("Features:", X.shape)
print("Missing values in X:", X.isna().sum().sum())
print("Missing target values:", y.isna().sum())
print("Target values:", sorted(y.unique()))

Cleaned dataset: (180519, 23)
Features: (180519, 17)
Missing values in X: 0
Missing target values: 0
Target values: [np.int64(0), np.int64(1)]


In [ ]:
## Train/Test Split

# We will simulate the real-world scenario:

# Past orders → Training data

# Future orders → Test data

# This is preferable to a random split for our first evaluation because
# the eventual system will predict future shipments.

In [ ]:
## Create Model Features and Target

# `X` contains the information that the model is allowed to use.

# `y` contains the variable we want the model to predict:

# `Late_delivery_risk`

# We keep the original `df_clean` dataframe intact and create
# separate objects for machine learning.

In [ ]:
X = df_clean[feature_cols].copy()
y = df_clean[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
for feature in feature_cols:
    print("-", feature)

X shape: (180519, 17)
y shape: (180519,)

Features:
- Type
- Shipping Mode
- Days for shipment (scheduled)
- Order_Month
- Order_DayOfWeek
- Order_Day
- Category Name
- Department Name
- Order Item Product Price
- Order Item Quantity
- Order Item Discount
- Order Item Discount Rate
- Customer Segment
- Market
- Order Region
- Order Country
- Order State


In [ ]:
print("Missing values in X:", X.isna().sum().sum())
print("Missing values in y:", y.isna().sum())

print("\nTarget values:")
print(sorted(y.unique()))

Missing values in X: 0
Missing values in y: 0

Target values:
[np.int64(0), np.int64(1)]


In [ ]:
model_df = df_clean[
    feature_cols + [TARGET, "order date (DateOrders)"]
].copy()

print("Model dataframe shape:", model_df.shape)

model_df.head()

Model dataframe shape: (180519, 19)


,Type,Shipping Mode,Days for shipment (scheduled),Order_Month,Order_DayOfWeek,Order_Day,Category Name,Department Name,Order Item Product Price,Order Item Quantity,Order Item Discount,Order Item Discount Rate,Customer Segment,Market,Order Region,Order Country,Order State,Late_delivery_risk,order date (DateOrders)
0,DEBIT,Standard Class,4,1,2,31,Sporting Goods,Fitness,327.75,1,13.110000,0.04,Consumer,Pacific Asia,Southeast Asia,Indonesia,Java Occidental,0,2018-01-31 22:56:00
1,TRANSFER,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,16.389999,0.05,Consumer,Pacific Asia,South Asia,India,Rajastán,1,2018-01-13 12:27:00
2,CASH,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,18.030001,0.06,Consumer,Pacific Asia,South Asia,India,Rajastán,0,2018-01-13 12:06:00
3,DEBIT,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,22.940001,0.07,Home Office,Pacific Asia,Oceania,Australia,Queensland,0,2018-01-13 11:45:00
4,PAYMENT,Standard Class,4,1,5,13,Sporting Goods,Fitness,327.75,1,29.500000,0.09,Corporate,Pacific Asia,Oceania,Australia,Queensland,0,2018-01-13 11:24:00


In [ ]:
# ## Chronological Ordering

# The real system will predict future shipments.

# Therefore, we should train on older orders and evaluate on newer
# orders rather than randomly mixing the past and future.

ORDER_DATE = "order date (DateOrders)"

model_df = (
    model_df
    .sort_values(ORDER_DATE)
    .reset_index(drop=True)
)

print(
    "Date range:",
    model_df[ORDER_DATE].min(),
    "→",
    model_df[ORDER_DATE].max()
)

Date range: 2015-01-01 00:00:00 → 2018-01-31 23:38:00


In [ ]:
split_index = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

print("Training rows:", len(train_df))
print("Testing rows :", len(test_df))

Training rows: 144415
Testing rows : 36104


In [ ]:
print("TRAINING PERIOD")
print(
    train_df[ORDER_DATE].min(),
    "→",
    train_df[ORDER_DATE].max()
)

print("\nTESTING PERIOD")
print(
    test_df[ORDER_DATE].min(),
    "→",
    test_df[ORDER_DATE].max()
)

TRAINING PERIOD
2015-01-01 00:00:00 → 2017-04-22 15:17:00

TESTING PERIOD
2017-04-22 15:17:00 → 2018-01-31 23:38:00


In [ ]:
X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (144415, 17)
y_train: (144415,)
X_test : (36104, 17)
y_test : (36104,)


In [ ]:
print("TRAINING TARGET")
print(
    y_train
    .value_counts()
    .sort_index()
)

print("\nTEST TARGET")
print(
    y_test
    .value_counts()
    .sort_index()
)

TRAINING TARGET
Late_delivery_risk
0    65171
1    79244
Name: count, dtype: int64

TEST TARGET
Late_delivery_risk
0    16371
1    19733
Name: count, dtype: int64


In [ ]:
print("TRAINING %")
print(
    y_train
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nTEST %")
print(
    y_test
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

TRAINING %
Late_delivery_risk
0    45.13
1    54.87
Name: proportion, dtype: float64

TEST %
Late_delivery_risk
0    45.34
1    54.66
Name: proportion, dtype: float64


In [ ]:
numeric_features = (
    X_train
    .select_dtypes(
        include=["int64", "float64"]
    )
    .columns
    .tolist()
)

print("Numerical features:")

for feature in numeric_features:
    print("-", feature)


categorical_features = (
    X_train
    .select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

print("\nCategorical features:")

for feature in categorical_features:
    print("-", feature)

Numerical features:
- Days for shipment (scheduled)
- Order Item Product Price
- Order Item Quantity
- Order Item Discount
- Order Item Discount Rate

Categorical features:
- Type
- Shipping Mode
- Category Name
- Department Name
- Customer Segment
- Market
- Order Region
- Order Country
- Order State


/var/folders/zs/hq3sw0fs2px2bvqk3dpf73sw0000gp/T/ipykernel_20480/4077107604.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(


In [ ]:
# ## Numerical Preprocessing

# For numerical features:

# 1. Missing values → median
# 2. Standardization → mean 0, standard deviation 1

# The preprocessing parameters will be learned ONLY from the
# training data.

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
# ## Categorical Preprocessing

# For categorical features:

# 1. Missing values → most frequent category
# 2. Categories → one-hot encoding

# `handle_unknown="ignore"` allows the model to handle a category
# that wasn't present during training.

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [ ]:
X_train_processed = (
    preprocessor
    .fit_transform(X_train)
)

X_test_processed = (
    preprocessor
    .transform(X_test)
)

print("Preprocessing complete.")

Preprocessing complete.


In [ ]:
print("Original X_train:", X_train.shape)
print("Processed X_train:", X_train_processed.shape)

print("\nOriginal X_test:", X_test.shape)
print("Processed X_test:", X_test_processed.shape)

Original X_train: (144415, 17)
Processed X_train: (144415, 1327)

Original X_test: (36104, 17)
Processed X_test: (36104, 1327)


In [ ]:
processed_feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Number of processed features:",
    len(processed_feature_names)
)

for feature in processed_feature_names[:50]:
    print(feature)

Number of processed features: 1327
numeric__Days for shipment (scheduled)
numeric__Order Item Product Price
numeric__Order Item Quantity
numeric__Order Item Discount
numeric__Order Item Discount Rate
categorical__Type_CASH
categorical__Type_DEBIT
categorical__Type_PAYMENT
categorical__Type_TRANSFER
categorical__Shipping Mode_First Class
categorical__Shipping Mode_Same Day
categorical__Shipping Mode_Second Class
categorical__Shipping Mode_Standard Class
categorical__Category Name_Accessories
categorical__Category Name_Baseball & Softball
categorical__Category Name_Boxing & MMA
categorical__Category Name_Camping & Hiking
categorical__Category Name_Cardio Equipment
categorical__Category Name_Cleats
categorical__Category Name_Electronics
categorical__Category Name_Fishing
categorical__Category Name_Fitness Accessories
categorical__Category Name_Girls' Apparel
categorical__Category Name_Golf Apparel
categorical__Category Name_Golf Balls
categorical__Category Name_Golf Gloves
categorical__Ca

In [ ]:
def count_nan(matrix):
    if hasattr(matrix, "toarray"):
        matrix = matrix.toarray()

    return np.isnan(matrix).sum()


print(
    "NaNs in training:",
    count_nan(X_train_processed)
)

print(
    "NaNs in testing:",
    count_nan(X_test_processed)
)

NaNs in training: 0
NaNs in testing: 0


In [ ]:
PREPROCESSOR_PATH = (
    MODEL_DIR
    / "preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH
)

print(
    "Preprocessor saved to:",
    PREPROCESSOR_PATH
)

Preprocessor saved to: /Users/Kushagra/Desktop/Code/LogiCast/src/models/preprocessor.joblib


In [ ]:
FEATURE_NAMES_PATH = (
    MODEL_DIR
    / "processed_feature_names.txt"
)

FEATURE_NAMES_PATH.write_text(
    "\n".join(processed_feature_names)
)

print(
    "Feature names saved to:",
    FEATURE_NAMES_PATH
)

Feature names saved to: /Users/Kushagra/Desktop/Code/LogiCast/src/models/processed_feature_names.txt


In [ ]:
print("=" * 55)
print("LogiCast — PREPROCESSING COMPLETE")
print("=" * 55)

print(f"Raw rows              : {len(df):,}")
print(f"Cleaned rows           : {len(df_clean):,}")
print(f"Training rows          : {len(X_train):,}")
print(f"Testing rows           : {len(X_test):,}")

print(f"\nOriginal features      : {X_train.shape[1]}")
print(
    f"Processed features     : "
    f"{X_train_processed.shape[1]}"
)

print(
    f"\nTraining period        : "
    f"{train_df[ORDER_DATE].min()} → "
    f"{train_df[ORDER_DATE].max()}"
)

print(
    f"Testing period         : "
    f"{test_df[ORDER_DATE].min()} → "
    f"{test_df[ORDER_DATE].max()}"
)

print(
    f"\nMissing values in X    : "
    f"{X.isna().sum().sum()}"
)

print(
    f"Missing values after preprocessing: "
    f"{count_nan(X_train_processed)}"
)

print(
    f"\nTarget classes         : "
    f"{sorted(y.unique())}"
)

print("=" * 55)

SUPPLYCHAIN SENTINEL — PREPROCESSING COMPLETE
Raw rows              : 180,519
Cleaned rows           : 180,519
Training rows          : 144,415
Testing rows           : 36,104

Original features      : 17
Processed features     : 1327

Training period        : 2015-01-01 00:00:00 → 2017-04-22 15:17:00
Testing period         : 2017-04-22 15:17:00 → 2018-01-31 23:38:00

Missing values in X    : 0
Missing values after preprocessing: 0

Target classes         : [np.int64(0), np.int64(1)]
